In [1]:
import os
os.chdir('/workspace/67f3128f-3156-42ba-bb9b-877561335f6b')
# List contents to discover what's available
for root, dirs, files in os.walk('.'):
 # avoid hidden
 dirs[:] = [d for d in dirs if not d.startswith('.')]
 level = root.replace('.', '').count(os.sep)
 indent = ' ' * 2 * level
 print(f'{indent}{os.path.basename(root)}/')
 subindent = ' ' * 2 * (level + 1)
 for f in files[:40]:
 try:
 size = os.path.getsize(os.path.join(root, f))
 except OSError:
 size = -1
 print(f'{subindent}{f} ({size} bytes)')
 if len(files) > 40:
 print(f'{subindent}... and {len(files)-40} more')


./
 .kernel_llm_logs_1.txt (0 bytes)
 -PROMPT-v6-DATASET.md (25848 bytes)
 memory/
 67f3128f-3156-42ba-bb9b-877561335f6b_memory_heron_c0_20260530T173205999930.jsonl (36727 bytes)


In [2]:
import mpmath as mp
mp.mp.dps = 50

# R2 validation: reproduce first three zeta zeros
g1 = mp.zetazero(1).imag
g2 = mp.zetazero(2).imag
g3 = mp.zetazero(3).imag
print(f"γ₁ = {mp.nstr(g1, 15)}")
print(f"γ₂ = {mp.nstr(g2, 15)}")
print(f"γ₃ = {mp.nstr(g3, 15)}")
expected = [14.134725141, 21.022039639, 25.010857580]
got = [g1, g2, g3]
for e, g in zip(expected, got):
 print(f"diff: {abs(float(g)-e):.3e}")


γ₁ = 14.1347251417347
γ₂ = 21.0220396387716
γ₃ = 25.0108575801457
diff: 7.347e-10
diff: 2.284e-10
diff: 1.457e-10


In [3]:
# Plan the work. Computing 5000 zeros for ζ via mpmath at dps=50 will be too slow.
# Time one zeta zero at dps=50.
import time
mp.mp.dps = 50
t0 = time.time()
z = mp.zetazero(1000)
print(z, time.time()-t0, "s")


(0.5 + 1419.4224809459956864659890380799168192321006010642j) 0.4443187713623047 s


In [4]:
t0 = time.time()
z = mp.zetazero(5000)
print(z, time.time()-t0, "s")


(0.5 + 5447.8619983012998564121586734642921676829672006076j) 0.9639737606048584 s


In [5]:
# 5000 zeros at dps=50 would take maybe 30-50 min for zeta. Let's plan smaller.
# Actually try moderate: full 5000 ζ zeros at dps=50 in parallel.
# Cost grows mildly. Estimate: average ~0.5s per zero -> 2500s ~ 42 min.
# That's borderline. Let me sample a few times.
ts = []
import time
mp.mp.dps = 50
for idx in [100, 500, 1000, 2000, 3000, 5000]:
 t0 = time.time()
 _ = mp.zetazero(idx)
 ts.append((idx, time.time()-t0))
print(ts)


[(100, 0.20911955833435059), (500, 0.31635332107543945), (1000, 0.4277794361114502), (2000, 0.5876214504241943), (3000, 0.801206111907959), (5000, 0.9887893199920654)]


In [6]:
# Let's estimate total time:
# integrate approx ~ 0.3 + 0.0001*n? Rough sum ≈ 5000*0.5 = 2500s. Too long for sequential.
# Use multiprocessing via concurrent.futures with mpmath. Since mpmath is pure Python,
# multi-CPU helps. Try with say 8 workers.
import os
print("CPU count:", os.cpu_count())


CPU count: 8


In [7]:
# Let's set up parallel computation of zeta zeros and L(χ4 mod 5) zeros and Δ zeros.
# Actually for Δ zeros and L_DH, we'd need different methods.
# Given constraints, delegate heavy computation to analysis sub-agent? Or just do here.
# Let's aim for: 
# - ζ: 5000 zeros at dps=50 (parallelized) ~ ~5-6 min
# - L(χ4 mod 5): 5000 zeros via mp.lzeros or similar 
# - L(Δ): use PARI/GP via cypari2 if available, else mpmath 
# - L_DH: 5000 zeros - need definition of L_DH. Let me check the spec.
# 
# We need L_DH's explicit definition (Davenport-Heilbronn?). Let me read the spec.


In [8]:
# The spec leaves L_DH and ζ_δ underspecified. Based on prior work documentation (mentioned in dataset description),
# L_DH refers to Davenport-Heilbronn function. The standard definition:
# Let θ = arctan(sqrt(10-2*sqrt(5))/(sqrt(5)-1))/2. Actually known: Davenport-Heilbronn is
# f(s) = (1-i tan θ)/2 * L(s, χ) + (1+i tan θ)/2 * L(s, χ̄)
# where χ is the non-principal character mod 5. With tan θ = (√10-2√5 - 2)/( √5 - 1) ... 
# Actually the canonical DH function is:
# f(s) = 5^(-s) * ((1-tan(θ)/(...))) ...
# Let me look up. Actually the standard form:
# f(s) = 1^(-s) + ξ * 2^(-s) - ξ * 3^(-s) - 4^(-s) + 0 * 5^(-s) + ...
# with ξ = (sqrt(10 - 2sqrt(5)) - 2) / (sqrt(5)-1) ≈ ... and period 5.
# 
# But computing 5000 zeros of L_DH is computationally prohibitive (dataset description says so).
# 
# Given the heavy computational cost (full 5000 zeros for 5 functions), I should delegate to analysis sub-agent.
# But this analysis is well-defined: compute Li coefficients from zero lists.
# 
# Let me look at what's feasible in this session. The research objective targets N=5000, dps=50.
# 
# Strategy:
# 1. Use mpmath.zetazero in parallel for ζ (and ζ_δ derived from ζ zeros).
# 2. Use mpmath for L(χ4 mod 5) zeros via mp.lerchphi/mp.dirichlet? mpmath has mp.zeros_lzeros? 
# Actually mpmath does NOT have a built-in Dirichlet L zero function. We need a custom solver.
# 3. L(Δ,s) zeros - LMFDB has tables.
# 4. L_DH zeros - sign change scan.
#
# Given time budget of 3600s and complexity, the responsible approach is to:
# - Compute fewer zeros (e.g., N=2000) practically, document this constraint.
# - Or delegate to sub-agent with longer compute time.
#
# Let me delegate to an analysis sub-agent with detailed instructions.
print("planning")


planning


In [9]:
# Actually let me reconsider. The task is computationally heavy but well-bounded. I have 3600s.
# Let me check if there are cached zero lists from previous runs.
import subprocess
r = subprocess.run(['find', '/workspace', '-name', '*.npy', '-o', '-name', '*.csv', '-o', '-name', '*zeros*'], 
 capture_output=True, text=True)
print(r.stdout[:5000])


In [10]:
# No cached files. Let's start by checking if Odlyzko's zeros are downloadable.
# Actually let's compute ζ zeros locally in parallel - that's the most we can scientifically attempt.
# Then handle the rest.
# 
# Given the constraints, my analysis approach will be:
# 1. Compute 5000 ζ zeros at dps=50 in parallel (estimate ~10-15 min wall clock with 8 cores).
# 2. Compute fewer L(χ) zeros — they're trickier without a library.
# 3. Use cached LMFDB tables for L(Δ) zeros if accessible.
# 4. For L_DH, the dataset description warns it's "particularly slow" — I'll attempt a small N.
# 5. Compute λ_n for n=1..200 using the formula λ_k = Σ_ρ [1 - (1 - 1/ρ)^k], summing over zeros and their conjugates.
#
# Honest scope: For zeros that take too long, I'll do a smaller N and report the limitation.
#
# Let me start by computing ζ zeros in parallel.
from concurrent.futures import ProcessPoolExecutor
import mpmath as mp

def zeta_zero_imag(n, dps=50):
 import mpmath as mp
 mp.mp.dps = dps
 z = mp.zetazero(n)
 return str(mp.mpf(z.imag)) # return as string for mpf serialization

# Test single call:
print(zeta_zero_imag(1))


14.134725141734693790457251983562470270784257115699


In [11]:
# Let me first do a smaller test - compute 200 zeros to estimate time
import time
from concurrent.futures import ProcessPoolExecutor

t0 = time.time()
with ProcessPoolExecutor(max_workers=8) as ex:
 results = list(ex.map(zeta_zero_imag, range(1, 201), chunksize=10))
t = time.time() - t0
print(f"200 zeros took {t:.1f}s; extrapolation for 5000: ~{t*25:.0f}s = {t*25/60:.1f} min")
print("first 3:", results[:3])


200 zeros took 7.5s; extrapolation for 5000: ~187s = 3.1 min
first 3: ['14.134725141734693790457251983562470270784257115699', '21.022039638771554992628479593896902777334340524903', '25.010857580145688763213790992562821818659549672558']


In [12]:
# Excellent! Faster than expected because of parallelism. Let's go for N=5000.
import time
t0 = time.time()
with ProcessPoolExecutor(max_workers=8) as ex:
 zeta_imag_strs = list(ex.map(zeta_zero_imag, range(1, 5001), chunksize=20))
t = time.time() - t0
print(f"5000 zeros took {t:.1f}s")
print("first 3:", zeta_imag_strs[:3])
print("last:", zeta_imag_strs[-1])


5000 zeros took 726.2s
first 3: ['14.134725141734693790457251983562470270784257115699', '21.022039638771554992628479593896902777334340524903', '25.010857580145688763213790992562821818659549672558']
last: 5447.8619983012998564121586734642921676829672006076


In [13]:
# Save zeros to disk
import os
os.makedirs('cache', exist_ok=True)
with open('cache/zeta_zeros_N5000_dps50.txt', 'w') as f:
 for s in zeta_imag_strs:
 f.write(s + '\n')
print("saved", len(zeta_imag_strs), "zeros")

# Convert to mpmath mpf
mp.mp.dps = 50
zeta_zeros_imag = [mp.mpf(s) for s in zeta_imag_strs]
print("min:", zeta_zeros_imag[0], "max:", zeta_zeros_imag[-1])

# Validation R2
print("\nR2 check:")
for i, expected in enumerate([14.134725141, 21.022039639, 25.010857580]):
 diff = abs(float(zeta_zeros_imag[i]) - expected)
 print(f" γ_{i+1}: got {float(zeta_zeros_imag[i]):.9f}, expected {expected}, diff={diff:.2e}")


saved 5000 zeros
min: 14.134725141734693790457251983562470270784257115699 max: 5447.8619983012998564121586734642921676829672006076

R2 check:
 γ_1: got 14.134725142, expected 14.134725141, diff=7.35e-10
 γ_2: got 21.022039639, expected 21.022039639, diff=2.28e-10
 γ_3: got 25.010857580, expected 25.01085758, diff=1.46e-10


In [14]:
# R2 passed for ζ. Now I need zeros for the other L-functions.
# 
# Time spent so far: ~12 min on zeros generation. Remaining budget ~ 48 min.
# 
# Strategy for remaining:
# - L(χ4 mod 5): mpmath has a Dirichlet L function via mp.dirichlet. We need zeros via mp.zero finder.
# Actually we can use mp.findroot on Z-function of the L. This is slower.
# Let me try a smaller N here.
# - L(Δ,s): try to fetch from LMFDB API.
# - L_DH: define the Davenport-Heilbronn function and find zeros via Hardy Z sign-change scan.
# - ζ_δ: derived from ζ zeros, so cheap.
#
# Given time constraints, I'll aim for:
# - 5000 ζ zeros (done) 
# - Try to fetch LMFDB Δ zeros (many available there)
# - For L(χ) and L_DH: compute as many zeros as time permits, document N achieved.
# - ζ_δ family from ζ zeros, all displacement levels.
#
# Given the substantial remaining work, I'll delegate further heavy compute (L(χ), L_DH, L(Δ) zeros) 
# to a data-retrieval/analysis sub-agent, while I prepare the analysis code in parallel.
#
# Actually let me think practically. Time has been used heavily. I should:
# 1. First try LMFDB for Δ zeros (free, just need URL).
# 2. For L(χ4 mod 5) - use mp.dirichlet + mp.findroot. 
# 3. For L_DH - sign change scan with the Davenport-Heilbronn definition.
# 
# Let me try LMFDB for Δ zeros via API.
from e14c.web import fetch_parsed_page
import asyncio
# LMFDB modular form 1.12.a.a is the Ramanujan Δ
# zeros endpoint: https://www.lmfdb.org/L/zeros/1.12.a.a (HTML) or downloadable


In [15]:
# Let me try the LMFDB API endpoint for L-function zeros
import urllib.request
import json

# Try direct download endpoint
urls_to_try = [
 "https://www.lmfdb.org/L/download_zeros/?label=4.1.1c1e2-1.1-c0e2.0-0", # not the right format
 "https://www.lmfdb.org/api/lfunctions/lfunctions?label=4-1-1.1-c11e2-0-0&_format=json",
 "https://www.lmfdb.org/L/Riemann/?download=zeros",
]
# Actually for modular form 1.12.a.a, the L-function page is /L/ModularForm/GL2/Q/holomorphic/1/12/a/a/
import urllib.request, urllib.error
req = urllib.request.Request("https://www.lmfdb.org/L/ModularForm/GL2/Q/holomorphic/1/12/a/a/", 
 headers={'User-Agent': 'Mozilla/5.0'})
try:
 resp = urllib.request.urlopen(req, timeout=30)
 body = resp.read().decode()
 print(body[:2000])
except Exception as e:
 print("err:", e)


<!DOCTYPE html>
<html lang="en">
 <head>
 <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
 <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
 <meta name="description" content="Welcome to the LMFDB, the database of L-functions, modular forms, and related objects. These pages are intended to be a modern handbook including tables, formulas, links, and references for L-functions and their underlying objects."/>
 <title>LMFDB - L-function 2-1-1.1-c11-0-0 </title>
 <link id="style_css" href="/style.css" rel="stylesheet" type="text/css" /> 

 <!-- jQuery -->
 <script type="text/javascript"
 src="https://ajax.googleapis.com/ajax/libs/jquery/3.2.1/jquery.min.js"
 integrity="sha384-xBuQ/xzmlsLoJpyjoggmTEz8OWUFM0/RC5BsqQBDX2v5cMvDHcMakNTNrHIW2I5f"
 crossorigin="anonymous"></script>
 <!-- jQuery UI -->
 <script type="text/javascript"
 src="https://ajax.googleapis.com/ajax/libs/jqueryui/1.12.1/jquery-ui.min.js"
 integrity="sha384-Dziy8F2VlJQLMShA6FHWNul/veM9bCkRUaLq

In [16]:
# Try the data download endpoint - LMFDB has a "Download zeros" link
req = urllib.request.Request("https://www.lmfdb.org/L/download_zeros/2-1-1.1-c11-0-0", 
 headers={'User-Agent': 'Mozilla/5.0'})
try:
 resp = urllib.request.urlopen(req, timeout=30)
 body = resp.read().decode()
 print(body[:3000])
except Exception as e:
 print("err:", e)



# Zeros of 2-1-1.1-c11-0-0 downloaded from the LMFDB on 30 May 2026.
{"order_of_vanishing": 0, "positive_zeros": ["9.222379399921102522243767192743", "13.90754986139213440644668132877", "17.44277697823447331355152513713", "19.65651314195496100012728175632", "22.33610363720986727568267445924", "25.27463654811236535674532419313", "26.80439115835040303257574923358", "28.83168262418687544502196191298", "31.17820949836025906449218889077", "32.77487538223120744183045567331"], "negative_zeros": ["-32.77487538223120744183045567331", "-31.17820949836025906449218889077", "-28.83168262418687544502196191298", "-26.80439115835040303257574923358", "-25.27463654811236535674532419313", "-22.33610363720986727568267445924", "-19.65651314195496100012728175632", "-17.44277697823447331355152513713", "-13.90754986139213440644668132877", "-9.222379399921102522243767192743"], "positive_zeros_accuracy": 100, "negative_zeros_accuracy": 100}


In [17]:
# Great, only 10 zeros from this endpoint though. Let's look for the full zero list endpoint.
# Try the API
for url in [
 "https://www.lmfdb.org/api/lfunc_lfunctions/?Lhash=2-1-1.1-c11-0-0",
 "https://www.lmfdb.org/L/2/1/1.1/c11/0/0",
 "https://www.lmfdb.org/L/Lfunction/2/1/1.1/c11/0/0",
 "https://beta.lmfdb.org/api/lfunc_lfunctions/?label=2-1-1.1-c11-0-0",
]:
 req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
 try:
 resp = urllib.request.urlopen(req, timeout=15)
 body = resp.read().decode()
 print(f"=== {url} ===")
 print(body[:500])
 print()
 except Exception as e:
 print(f"err for {url}: {e}")


=== https://www.lmfdb.org/api/lfunc_lfunctions/?Lhash=2-1-1.1-c11-0-0 ===
<!DOCTYPE html>
<html lang="en">
 <head>
 <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
 <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
 <meta name="description" content="Welcome to the LMFDB, the database of L-functions, modular forms, and related objects. These pages are intended to be a modern handbook including tables, formulas, links, and references for L-functions and their underlying objects."/>
 <title>LMFDB - Database - lfunc_lfun



=== https://www.lmfdb.org/L/2/1/1.1/c11/0/0 ===
<!DOCTYPE html>
<html lang="en">
 <head>
 <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
 <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
 <meta name="description" content="Welcome to the LMFDB, the database of L-functions, modular forms, and related objects. These pages are intended to be a modern handbook including tables, formulas, links, and references for L-functions and their underlying objects."/>
 <title>LMFDB - L-function 2-1-1.1-c1

err for https://www.lmfdb.org/L/Lfunction/2/1/1.1/c11/0/0: HTTP Error 404: Not Found


=== https://beta.lmfdb.org/api/lfunc_lfunctions/?label=2-1-1.1-c11-0-0 ===
<!DOCTYPE html>
<html>
<meta charset="UTF-8">
<noscript><p>Enable JavaScript, then reload.</p></noscript>
<script>
 // Set cookie (expires in 1 day)
 document.cookie = "human=1; path=/; max-age=86400";
 // Get the 'gateorig' parameter from URL
 var params = new URLSearchParams(window.location.search);
 if (params.has('gateorig')) {
 var orig = params.get('gateorig');
 params.delete('gateorig');
 } else {
 var orig = '/';
 }
 params = params.toString();
 if (params) {
 



In [18]:
# LMFDB only easily gives us 10 zeros. For 5000 Δ zeros, we'd need PARI/GP's lfunzeros (as noted in dataset description).
# Let me check if pari/cypari2 is available.
try:
 import cypari2
 print("cypari2 available")
except ImportError:
 print("cypari2 NOT available")
try:
 import subprocess
 r = subprocess.run(['gp', '--version'], capture_output=True, text=True, timeout=5)
 print("gp:", r.stdout[:200], r.stderr[:200])
except Exception as e:
 print("no gp:", e)


cypari2 NOT available
no gp: [Errno 13] Permission denied: 'gp'


In [19]:
r = subprocess.run(['which', 'gp'], capture_output=True, text=True)
print(r.stdout, r.stderr)
r = subprocess.run(['pip', 'install', 'cypari2'], capture_output=True, text=True, timeout=120)
print(r.stdout[-500:])
print("STDERR:", r.stderr[-500:])


ysignals>=1.11.3 (from cypari2)

STDERR: 


In [20]:
import cypari2
pari = cypari2.Pari()
# Test
print(pari("Pi"))
# Compute zeros of L(Δ,s). The L-function of the Ramanujan Δ has weight 12, level 1.
# In PARI/GP: L = lfunmf(mfinit([1,12],0)[1]) ; lfunzeros(L, max_T)
# Let me try
try:
 res = pari("M = mfinit([1,12], 0); F = mfeigenbasis(M)[1]; L = lfunmf(F); lfunzeros(L, 20)")
 print(res)
except Exception as e:
 print("err:", e)


3.14159265358979
err: lfunmf: incorrect type in checkMF [please use mfinit] (t_VEC)


In [21]:
# mfinit returns a vector; we need to pick the form differently
try:
 res = pari("M = mfinit([1,12], 1); F = mfeigenbasis(M)[1]; L = lfunmf([M,F]); lfunzeros(L, 50)")
 print(res)
except Exception as e:
 print("err:", e)


err: lfunmf: incorrect type in checkMF [please use mfinit] (t_VEC)


In [22]:
# Try alternative API
for cmd in [
 "M = mfinit([1,12], 1); F = mfeigenbasis(M)[1]; L = lfunmf(M, F); lfunzeros(L, 50)",
 "L = lfunmf(mfinit([1,12,1])); lfunzeros(L, 50)",
 "L = lfuncreate(mfinit([1,12,1])); lfunzeros(L, 50)",
 "L = lfunramanujan(); lfunzeros(L, 50)",
]:
 try:
 res = pari(cmd)
 print(cmd, "->", res)
 break
 except Exception as e:
 print("err:", cmd, "->", e)


M = mfinit([1,12], 1); F = mfeigenbasis(M)[1]; L = lfunmf(M, F); lfunzeros(L, 50) -> [9.22237939992110, 13.9075498613921, 17.4427769782345, 19.6565131419550, 22.3361036372099, 25.2746365481124, 26.8043911583504, 28.8316826241869, 31.1782094983603, 32.7748753822312, 35.1969958412101, 36.7414629767103, 37.7539159756243, 40.2190343742213, 41.7304922893078, 43.5917412355752, 45.0400792137756, 46.1973187531433, 48.3590524780237, 49.2760535365582]


In [23]:
# Excellent! PARI works. Now let's get many more zeros.
# lfunzeros(L, T) gives zeros up to imaginary part T.
# We want ~5000 zeros. Let's estimate: density of zeros at height T is ~ d * log(T)/(2π) per unit T, where d=2 for degree 2.
# So 5000 zeros => need T such that integral ~ 5000 => T ≈ ~700-900? Let's try in chunks.

import time
t0 = time.time()
res = pari("M = mfinit([1,12], 1); F = mfeigenbasis(M)[1]; L = lfunmf(M, F); v = lfunzeros(L, 200); #v")
print("zeros up to T=200:", res, "in", time.time()-t0, "s")


zeros up to T=200: 159 in 0.5104436874389648 s


In [24]:
# 159 zeros up to T=200. So 5000 zeros need T ≈ scaled.
# T=200 -> 159 zeros. Density grows logarithmically. Try T=4000.
t0 = time.time()
n_zeros = int(pari("M = mfinit([1,12], 1); F = mfeigenbasis(M)[1]; L = lfunmf(M, F); v = lfunzeros(L, 4000); #v"))
print("zeros up to T=4000:", n_zeros, "in", time.time()-t0, "s")


 *** lfunzeros: Warning: increasing stack size to 8003584.


PariError: lfunzeros: the PARI stack overflows (current size: 8003584; maximum size: 8003584)
You can use pari.allocatemem() to change the stack size and try again

In [25]:
pari.allocatemem(2**30) # 1 GB
t0 = time.time()
n_zeros = int(pari("M = mfinit([1,12], 1); F = mfeigenbasis(M)[1]; L = lfunmf(M, F); v = lfunzeros(L, 2000); #v"))
print(f"Δ zeros up to T=2000:", n_zeros, "in", time.time()-t0, "s")


TimeoutError: Code execution timed out after 900.0 seconds

In [26]:
# Took too long. Need to allocate properly and use smaller T then go incrementally.
import cypari2
pari = cypari2.Pari()
pari.allocatemem(2**30)
t0 = time.time()
n_zeros = int(pari("M = mfinit([1,12], 1); F = mfeigenbasis(M)[1]; L = lfunmf(M, F); v = lfunzeros(L, 500); #v"))
print(f"Δ zeros up to T=500:", n_zeros, "in", time.time()-t0, "s")


In [27]:
# Cell didn't output. Let me proceed pragmatically given limited time.
# Reset PARI - it took too long.
# Given remaining ~17 min, I will produce results for the controls I can actually compute:
# - ζ: 5000 zeros (have them) — strongest result
# - ζ_δ deformation: derived from ζ zeros (cheap) 
# - L(Δ,s): use the 20 zeros we already retrieved from LMFDB (small but rigorous)
# - L(χ4 mod 5): small N via mpmath (limited)
# - L_DH: skip due to definition ambiguity & compute cost; document this
#
# Plan: compute λ_n for n=1..200 from each available zero list.

import mpmath as mp
mp.mp.dps = 50

# Re-read cached zeta zeros
with open('cache/zeta_zeros_N5000_dps50.txt') as f:
 zeta_imag_strs = [line.strip() for line in f if line.strip()]
zeta_zeros_imag = [mp.mpf(s) for s in zeta_imag_strs]
print("Have", len(zeta_zeros_imag), "ζ zeros")


Have 5000 ζ zeros


In [28]:
# Implement Li coefficient computation.
# λ_k = Σ_ρ [1 - (1 - 1/ρ)^k]
# Sum over ALL non-trivial zeros (both ρ and conjugate -ρ̄ = ρ̄ pair). For ζ, since zeros come in
# conjugate pairs ρ = 1/2 + iγ and ρ̄ = 1/2 - iγ, the symmetric sum gives real λ_k.
# For each zero ρ on the critical line, we add contributions from both ρ and ρ̄.

import numpy as np
import mpmath as mp

def li_coefficients(zeros_complex, K=200, dps=50, sym_pair=True):
 """
 Compute λ_1..λ_K from list of complex zeros (upper half-plane).
 If sym_pair=True, automatically include conjugate -bar(ρ) = ρ̄.
 Formula: λ_k = Σ_ρ [1 - (1 - 1/ρ)^k]
 """
 mp.mp.dps = dps
 K = int(K)
 lambdas = [mp.mpf(0) for _ in range(K+1)] # 1..K used
 one = mp.mpf(1)
 for rho in zeros_complex:
 # u = 1 - 1/ρ (then 1 - u^k contributes)
 inv_rho = one/rho
 u = one - inv_rho
 # accumulate 1 - u^k for k=1..K
 uk = one
 for k in range(1, K+1):
 uk = uk * u
 lambdas[k] += (one - uk)
 if sym_pair:
 rho_c = mp.mpc(rho.real, -rho.imag)
 inv_rho_c = one/rho_c
 uc = one - inv_rho_c
 uck = one
 for k in range(1, K+1):
 uck = uck * uc
 lambdas[k] += (one - uck)
 # return real parts as floats (and full mpf for precision)
 return lambdas[1:]

# Test on first 3 ζ zeros — these should reproduce known λ_n values approximately
zs_test = [mp.mpc(0.5, g) for g in zeta_zeros_imag[:3]]
res = li_coefficients(zs_test, K=5, dps=50)
for k, v in enumerate(res, 1):
 print(f"λ_{k} (test, N=3) = {mp.nstr(v.real, 10)} + {mp.nstr(v.imag, 5)}i")


λ_1 (test, N=3) = 0.008858503479 + 0.0i
λ_2 (test, N=3) = 0.03540135594 + 0.0i
λ_3 (test, N=3) = 0.07953072401 + 0.0i
λ_4 (test, N=3) = 0.14108402 + 0.0i
λ_5 (test, N=3) = 0.2198346015 + 0.0i


In [29]:
# Good - values are positive and small as expected for partial sum.
# Known λ_1 (full) = (γ + log(4π) - 2)/2 ≈ 0.0230957 -- our N=3 partial is smaller.
# That matches expected behavior.
#
# Now do the full computation for ζ with N=5000 and N=2000.
# But mpmath loop with K=200 and N=5000 zeros -> 5000*200*2 = 2,000,000 mpf multiplications.
# At dps=50 each is slow. Estimate ~5-10 min. Tight.
#
# Use numpy double precision instead - we need to check precision tradeoffs.
# For λ_n with n up to 200 and zeros having |γ|~5000, |1-1/ρ|^200 is fine.
# Let's check what precision we need. (1-1/ρ) for ρ=0.5+iγ small γ
# |1-1/ρ|^2 = ((|ρ|^2 - 1)^2 + γ^2*... ). For γ=14, |1-1/ρ| ≈ |1-1/(0.5+14i)| ≈ 0.9974
# After 200 powers: 0.9974^200 ≈ 0.59. OK.
# For γ large, |ρ| large, 1-1/ρ close to 1. (1-1/ρ)^200 ≈ 1, so 1 - (1-1/ρ)^200 small. 
# Cancellation matters - but at most ~ 200/|ρ| ≈ 0.04 for γ=5000.
# Total sum should be accurate enough in double precision for trend.
#
# Use double precision (numpy/Python complex) but with mpmath as final accuracy check on subset.

import numpy as np

def li_coefficients_np(zeros_complex_np, K=200):
 """zeros_complex_np: numpy complex128 array of zeros (upper half). Returns array of length K with λ_1..λ_K."""
 inv = 1.0 / zeros_complex_np
 u = 1.0 - inv # complex array, shape (N,)
 # conjugate too
 zeros_c = np.conjugate(zeros_complex_np)
 u_c = 1.0 - 1.0/zeros_c
 
 lambdas = np.zeros(K, dtype=np.complex128)
 # u^k for k=1..K iteratively
 uk = np.ones_like(u)
 uck = np.ones_like(u_c)
 for k in range(1, K+1):
 uk *= u
 uck *= u_c
 s = np.sum(1.0 - uk) + np.sum(1.0 - uck)
 lambdas[k-1] = s
 return lambdas

# ζ zeros as complex array
zeta_zeros_complex = np.array([complex(0.5, float(g)) for g in zeta_zeros_imag], dtype=np.complex128)
print("shape:", zeta_zeros_complex.shape)

# Test reproducibility
import time
t0 = time.time()
lam_zeta_5000 = li_coefficients_np(zeta_zeros_complex, K=200)
print(f"ζ N=5000, K=200: {time.time()-t0:.2f}s")
print("First 10 λ_n:")
for k in range(1, 11):
 print(f" λ_{k} = {lam_zeta_5000[k-1].real:.10f} (imag {lam_zeta_5000[k-1].imag:.2e})")


shape: (5000,)
ζ N=5000, K=200: 0.01s
First 10 λ_n:
 λ_1 = 0.0228688736 (imag 0.00e+00)
 λ_2 = 0.0914383936 (imag 0.00e+00)
 λ_3 = 0.2055974018 (imag 0.00e+00)
 λ_4 = 0.3651611129 (imag 0.00e+00)
 λ_5 = 0.5698718292 (imag 0.00e+00)
 λ_6 = 0.8193999376 (imag 0.00e+00)
 λ_7 = 1.1133451827 (imag 0.00e+00)
 λ_8 = 1.4512382114 (imag 0.00e+00)
 λ_9 = 1.8325423810 (imag 0.00e+00)
 λ_10 = 2.2566558236 (imag 0.00e+00)


In [30]:
# Excellent! Very fast in numpy. λ_1 ≈ 0.0228688... compared to the true value
# λ_1 = (γ_Euler + log(4π) - 2)/2 ≈ 0.0230957 — our partial sum is close.
# Known value of λ_2 ≈ 0.0923457
# So partial sums underestimate slightly.

# Now do all required computations: 
# - ζ N=5000 (done)
# - ζ N=2000 (for truncation check)
# - ζ_δ for δ ∈ {1e-3, 1e-2, 1e-1} and m ∈ {1, 5, 20} 
# - L(Δ): only 20 zeros available
# - L(χ_4 mod 5): need to compute. Skip for time?
# - L_DH: skip due to definition ambiguity

# Truncation check ζ N=2000
lam_zeta_2000 = li_coefficients_np(zeta_zeros_complex[:2000], K=200)
rel_diff_zeta = np.abs(lam_zeta_5000 - lam_zeta_2000) / np.abs(lam_zeta_5000)
print("Truncation check (ζ N=5000 vs N=2000):")
for k in [1, 5, 10, 50, 100, 150, 200]:
 print(f" λ_{k}: |Δ|/|λ| = {rel_diff_zeta[k-1]:.3e}")
print(f" max rel diff: {rel_diff_zeta.max():.3e} at k={rel_diff_zeta.argmax()+1}")


Truncation check (ζ N=5000 vs N=2000):
 λ_1: |Δ|/|λ| = 9.423e-03
 λ_5: |Δ|/|λ| = 9.454e-03
 λ_10: |Δ|/|λ| = 9.550e-03
 λ_50: |Δ|/|λ| = 1.254e-02
 λ_100: |Δ|/|λ| = 1.852e-02
 λ_150: |Δ|/|λ| = 2.393e-02
 λ_200: |Δ|/|λ| = 2.896e-02
 max rel diff: 2.896e-02 at k=200


In [31]:
# All λ_n positive for ζ. Relative diff between N=2000 and N=5000 is ~1-3%, reasonable.
# Now ζ_δ deformation - displace m zeros off the line by δ.

def make_zeta_delta_zeros(zeros_complex, delta, m):
 """Displace first m zeros by δ (horizontal shift Re part)."""
 zs = zeros_complex.copy()
 # Shift first m zeros: ρ = 0.5 + iγ -> (0.5 + δ) + iγ. This creates off-line zeros.
 # By functional equation symmetry, also should have a partner at (0.5 - δ) + iγ.
 # We'll use a "one-sided" displacement (as documented in dataset description r3).
 zs[:m] = (0.5 + delta) + 1j * zs[:m].imag
 return zs

# Compute λ_n for ζ_δ family
delta_results = {}
for delta in [0.0, 1e-3, 1e-2, 1e-1]:
 for m in [1, 5, 20]:
 zs = make_zeta_delta_zeros(zeta_zeros_complex, delta, m)
 lam = li_coefficients_np(zs, K=200)
 delta_results[(delta, m)] = lam
 # Check positivity
 re = lam.real
 n_neg = np.sum(re < 0)
 idx_neg = np.where(re < 0)[0] + 1
 print(f" δ={delta}, m={m}: any negative λ_n? {n_neg>0}, count={n_neg}, indices={list(idx_neg[:10])}")


 δ=0.0, m=1: any negative λ_n? False, count=0, indices=[]
 δ=0.0, m=5: any negative λ_n? False, count=0, indices=[]
 δ=0.0, m=20: any negative λ_n? False, count=0, indices=[]
 δ=0.001, m=1: any negative λ_n? False, count=0, indices=[]
 δ=0.001, m=5: any negative λ_n? False, count=0, indices=[]
 δ=0.001, m=20: any negative λ_n? False, count=0, indices=[]
 δ=0.01, m=1: any negative λ_n? False, count=0, indices=[]
 δ=0.01, m=5: any negative λ_n? False, count=0, indices=[]
 δ=0.01, m=20: any negative λ_n? False, count=0, indices=[]
 δ=0.1, m=1: any negative λ_n? False, count=0, indices=[]
 δ=0.1, m=5: any negative λ_n? False, count=0, indices=[]
 δ=0.1, m=20: any negative λ_n? False, count=0, indices=[]


In [32]:
# For our one-sided displacement, none became negative for these δ/m within n≤200.
# The deformation is asymmetric, may not be enough perturbation. Document this.

# Now L(Δ,s) using the 20 zeros we have from LMFDB.
delta_zeros_T = [
 9.222379399921102522243767192743, 13.90754986139213440644668132877,
 17.44277697823447331355152513713, 19.65651314195496100012728175632,
 22.33610363720986727568267445924, 25.27463654811236535674532419313,
 26.80439115835040303257574923358, 28.83168262418687544502196191298,
 31.17820949836025906449218889077, 32.77487538223120744183045567331,
 35.1969958412101, 36.7414629767103, 37.7539159756243, 40.2190343742213,
 41.7304922893078, 43.5917412355752, 45.0400792137756, 46.1973187531433,
 48.3590524780237, 49.2760535365582,
]
delta_zeros_complex = np.array([complex(0.5, g) for g in delta_zeros_T], dtype=np.complex128)
lam_delta = li_coefficients_np(delta_zeros_complex, K=200)
re = lam_delta.real
n_neg = int(np.sum(re < 0))
print(f"L(Δ,s) N={len(delta_zeros_complex)}: λ_1={re[0]:.5f}, λ_200={re[-1]:.5f}")
print(f" Any negative? {n_neg>0} (count {n_neg})")
print(f" λ trend: first 5 = {re[:5]}")
print(f" λ trend: last 5 = {re[-5:]}")


L(Δ,s) N=20: λ_1=0.03665, λ_200=37.60080
 Any negative? False (count 0)
 λ trend: first 5 = [0.03664667 0.14638986 0.32864089 0.58242472 0.90638892]
 λ trend: last 5 = [38.01997399 37.94092653 37.84585166 37.73298386 37.60079837]


In [33]:
# L(Δ) with only N=20 zeros - λ_n positive but limited convergence.
# Note: for L(Δ) on critical line Re=1/2 (analytic normalization), all λ_n should be ≥ 0 under GRH.

# Now L(χ_4 mod 5). The character mod 5 with order 4 is the unique primitive Dirichlet character mod 5
# with χ(2) = i (a primitive character of order 4 modulo 5).
# χ_5(n): χ(1)=1, χ(2)=i, χ(3)=-i, χ(4)=-1, χ(5)=0, period 5.

# Use mpmath's dirichlet
import mpmath as mp

def chi5(n):
 n = n % 5
 table = {1: 1, 2: 1j, 3: -1j, 4: -1, 0: 0}
 return table[n]

# Test L(s, χ) using mp.dirichlet — actually use partial summation. For finding zeros, 
# use mp.findroot on the Hardy-Z function. But this is complex Dirichlet L.
# Easier: directly call mp.zeta-like for L. mpmath has no built-in. Need custom Z-function.
# 
# Given remaining ~16 min, I'll skip L(χ) computation and use a small sample approach OR 
# use known facts. Actually mp has lerchphi - too involved.
# 
# Best option: use PARI's lfunzeros on Dirichlet character.
import time
t0 = time.time()
try:
 res = pari("L = lfuncreate([5, [0,1,-1,0]]); v = lfunzeros(L, 500); #v")
 print(f"L(χ4 mod 5) zeros up to T=500: count={int(res)} in {time.time()-t0:.1f}s")
except Exception as e:
 print("err:", e)


err: lfuncreate: incorrect type in lfunmisc_to_ldata (t_VEC)


In [34]:
# Try via znchar - PARI uses znstar for character
import time
t0 = time.time()
try:
 # Create the primitive order-4 character mod 5
 res = pari("G = znstar(5, 1); chi = [G, [1]]; L = lfuncreate(chi); v = lfunzeros(L, 500); v[1..min(#v,5)]")
 print(f"L(χ4 mod 5) test in {time.time()-t0:.1f}s:")
 print(res)
except Exception as e:
 print("err:", e)


L(χ4 mod 5) test in 2.3s:
[6.18357819545085, 8.45722917442323, 12.6749464170114, 14.8250255703284, 17.3378021068530]


In [35]:
# Now go larger
import time
pari.allocatemem(2**30)
t0 = time.time()
res = pari("G = znstar(5, 1); chi = [G, [1]]; L = lfuncreate(chi); v = lfunzeros(L, 1500); v")
elapsed = time.time()-t0
zeros_chi = [float(x) for x in res]
print(f"L(χ4 mod 5): {len(zeros_chi)} zeros up to T=1500 in {elapsed:.1f}s")
print("first 3:", zeros_chi[:3])


PARI stack size set to 1073741824 bytes, maximum size set to 1073741824
L(χ4 mod 5): 1450 zeros up to T=1500 in 80.0s
first 3: [6.1835781954508535, 8.457229174423231, 12.674946417011355]


In [36]:
# Got 1450 zeros. That's good. For complex L(s, χ), zeros do NOT come in conjugate pairs in the same simple way!
# Wait — chi is non-real (complex character). Then L(s, χ̄) has zeros at conjugate positions.
# However, the Li coefficient is typically defined for self-dual L-functions or uses 
# the symmetric pairing ρ ↔ 1-ρ̄. For a non-real character, the completed Λ(s, χ) satisfies
# Λ(s, χ) = ε Λ(1-s, χ̄), so zeros of L(s,χ) at ρ pair with zeros of L(s,χ̄) at 1-ρ̄.
#
# Standard Li coefficient for L(s,χ) (non-real character) uses both L(s,χ) and L(s,χ̄):
# λ_n(χ) = (1/2) Σ_{ρ of L(s,χ)} [1 - (1-1/ρ)^n] + (1/2) Σ_{ρ of L(s,χ̄)} [1 - (1-1/ρ)^n]
# Equivalently: pair each zero of L(s,χ) with conjugate zeros via χ̄.
# 
# But for χ_4 mod 5 (order 4, complex), L(s,χ̄) = conjugate of L(s,χ), so zeros of L(s,χ̄) 
# are conjugates of zeros of L(s,χ). On the critical line, zeros of L(s,χ) are at 1/2 + iγ_n
# for both positive and negative γ. PARI returns POSITIVE γ only typically.
# Including the conjugate zeros (1/2 - iγ) will make the Li coefficient real.
#
# I'll use the symmetric pairing — sum over zeros and their complex conjugates, as for ζ.
# This is appropriate for the standard "completed" Li coefficient.

chi_zeros_complex = np.array([complex(0.5, g) for g in zeros_chi], dtype=np.complex128)
lam_chi = li_coefficients_np(chi_zeros_complex, K=200)
re = lam_chi.real
n_neg = int(np.sum(re < 0))
print(f"L(χ4 mod 5) N={len(chi_zeros_complex)}: λ_1={re[0]:.5f}, λ_200={re[-1]:.5f}")
print(f" Any negative? {n_neg>0} (count {n_neg})")
print(f" imag part max: {np.abs(lam_chi.imag).max():.2e}")
print(f" first 5: {re[:5]}")
print(f" last 5: {re[-5:]}")


L(χ4 mod 5) N=1450: λ_1=0.08495, λ_200=429.48748
 Any negative? False (count 0)
 imag part max: 0.00e+00
 first 5: [0.08495187 0.33884218 0.75879566 1.34008859 2.07624964]
 last 5: [419.71177476 422.10374182 424.5309982 426.99283083 429.48748163]


In [37]:
# Now L_DH. Davenport-Heilbronn: standard definition 
# f(s) = (5/π)^s ζ(s, 1/5) * (1+i tan θ)/2 * ... 
# The most common form: 
# f(s) = (5^(-s)/2) * [(1-i ξ) Φ(s,1) + (1+i ξ) Φ(s,2)]
# where ξ = (sqrt(10 - 2*sqrt(5)) - 2)/(sqrt(5) - 1), Φ(s,a) = ζ(s, a/5) Hurwitz.
# A cleaner equivalent representation: 
# f(s) = 1^{-s} + ξ·2^{-s} - ξ·3^{-s} - 4^{-s} + 0·5^{-s}, period 5, ξ ≈ ?
# 
# Standard ξ value: ξ = (√10 - 2√5 - 2)/(√5 - 1)
import mpmath as mp
mp.mp.dps = 30
ksi = (mp.sqrt(10 - 2*mp.sqrt(5)) - 2)/(mp.sqrt(5) - 1)
print("ξ =", ksi)

# Test the off-line zeros from R2
def L_DH_partial(s, N=200000):
 """Compute Davenport-Heilbronn L-function via approximate functional equation.
 Period 5 sequence: c = [1, ξ, -ξ, -1, 0]
 f(s) = Σ_{n=1}^∞ c(n) / n^s
 """
 s = mp.mpc(s)
 c = [mp.mpf(1), ksi, -ksi, mp.mpf(-1), mp.mpf(0)]
 total = mp.mpc(0)
 for n in range(1, N+1):
 coef = c[(n-1) % 5]
 if coef == 0:
 continue
 total += coef / mp.power(n, s)
 return total

# Need a functional equation form. For zeros at large height, direct partial sum is bad.
# Use Hurwitz zeta representation:
def L_DH(s):
 """L_DH(s) using Hurwitz zeta representation - converges for all s with Re(s) > 0."""
 # f(s) = 5^{-s} Σ_{a=1}^5 c(a) ζ(s, a/5)
 s = mp.mpc(s)
 c = [mp.mpf(1), ksi, -ksi, mp.mpf(-1), mp.mpf(0)]
 total = mp.mpc(0)
 for a in range(1, 6):
 total += c[a-1] * mp.zeta(s, mp.mpf(a)/5)
 return mp.power(5, -s) * total

# Test at the four R2 off-line zeros
mp.mp.dps = 30
off_line = [(0.808517, 85.699348), (0.650786, 114.163343), (0.574355, 166.479306), (0.724258, 176.702461)]
for sigma, t in off_line:
 val = L_DH(mp.mpc(sigma, t))
 print(f" L_DH({sigma}, {t}) = {val} |.|={abs(val):.3e}")


ξ = 0.284079043840412296028291832393


TypeError: unsupported format string passed to mpf.__format__

In [38]:
for sigma, t in off_line:
 val = L_DH(mp.mpc(sigma, t))
 print(f" L_DH({sigma}, {t}) = {val}, |L_DH|={float(abs(val)):.3e}")


 L_DH(0.808517, 85.699348) = (-0.000000342083505722335086204429306202 - 0.000000554060837117548499854948460082j), |L_DH|=6.512e-07
 L_DH(0.650786, 114.163343) = (-0.0000383348627610133911017773187976 - 0.0000125397200588541223483443584063j), |L_DH|=4.033e-05
 L_DH(0.574355, 166.479306) = (-0.000000632460835063346115506750658781 + 0.000000253974756907476968009778763578j), |L_DH|=6.815e-07
 L_DH(0.724258, 176.702461) = (0.000000283141634924537449006983550324 - 0.000000376086153022667363419509737074j), |L_DH|=4.708e-07


In [39]:
# 3 of 4 off-line zeros pass <10^-6. The (0.650786, 114.163343) is 4e-5 (matches dataset note).
# R2 functionally passed for L_DH.
# 
# Now finding zeros of L_DH. The dataset description says PARI's lfunzeros misses zeros, 
# and the correct method is a Hardy-Z function sign-change scan. This is computationally expensive.
# Given remaining time ~14 min, I'll do a small sign-change scan to find ~50-100 zeros 
# on the critical line, which is enough to test λ_n sign and demonstrate the comparison.
#
# To find zeros on the critical line, we use the Z-function approach. For non-trivial gamma factor,
# we'd need the exact functional equation. Since L_DH satisfies its own functional equation 
# relating s ↔ 1-s with conductor 5/π gamma factor (similar to Dirichlet L mod 5 with χ4),
# we can just look for sign changes of Im(L_DH(1/2 + it)) ... but L_DH is complex-valued on the critical line generally.
#
# Practical approach: scan |L_DH(1/2 + it)| for local minima close to zero.
# Better: since L_DH is sum of two Dirichlet L(s, χ), its zeros on critical line correspond to 
# real Z-function. Skip explicit Hardy-Z, just look for |L_DH(1/2+it)|<eps and refine.
#
# Given time, I'll do a coarse scan and find ~50 zeros to demonstrate.

import mpmath as mp
mp.mp.dps = 25

def L_DH_half(t):
 return L_DH(mp.mpc(0.5, t))

import time
t0 = time.time()
# Coarse scan up to T=200
ts_scan = np.linspace(0.1, 200, 4000)
vals = []
for t in ts_scan:
 v = L_DH_half(t)
 vals.append(complex(v))
vals = np.array(vals)
print(f"scan to T=200 in {time.time()-t0:.1f}s")


scan to T=200 in 78.9s


In [40]:
# Use real and imag sign changes to find on-line zeros
# L_DH on critical line: at on-line zeros both Re=0 and Im=0 simultaneously
# Use sign change of "Hardy Z-like" function. For Davenport-Heilbronn function f(s),
# define χ(s) from functional equation: f(s) = χ(s) f(1-s), then Z(t) = χ(1/2+it)^(-1/2) f(1/2+it) is real-valued
# Without the gamma factor, simpler: find local minima of |L_DH|^2.

# Find local minima of |vals|
absvals = np.abs(vals)
mins_idx = []
for i in range(2, len(vals)-2):
 if absvals[i] < absvals[i-1] and absvals[i] < absvals[i+1] and absvals[i] < 0.05:
 mins_idx.append(i)
print(f"Local minima with |L_DH|<0.05: {len(mins_idx)}")
# Refine using mp.findroot to find on-line zeros
def fdh_real_imag(t):
 v = L_DH(mp.mpc(0.5, t))
 return [float(v.real), float(v.imag)]

dh_zeros = []
seen = set()
for i in mins_idx[:80]: # cap for time
 t_guess = ts_scan[i]
 # Use Newton-like via mp.findroot for real and imag both zero
 try:
 mp.mp.dps = 20
 # Find zero of |L_DH|^2 derivative? Easier: assume on critical line means Re and Im vanish.
 # Use 2-d findroot
 t_refined = mp.findroot(lambda t: L_DH(mp.mpc(0.5, t)), t_guess, solver='secant', tol=1e-12)
 t_val = float(mp.mpf(t_refined).real if hasattr(t_refined,'real') else t_refined)
 # Check it's a true zero
 v = abs(L_DH(mp.mpc(0.5, t_val)))
 if float(v) < 1e-6 and t_val > 0:
 t_round = round(t_val, 4)
 if t_round not in seen:
 seen.add(t_round)
 dh_zeros.append(t_val)
 except Exception as e:
 pass

dh_zeros = sorted(dh_zeros)
print(f"Found {len(dh_zeros)} L_DH on-line zeros up to T~200")
print("first 10:", dh_zeros[:10])
print("last 5:", dh_zeros[-5:])


Local minima with |L_DH|<0.05: 73


Found 0 L_DH on-line zeros up to T~200
first 10: []
last 5: []


In [41]:
# findroot didn't converge well. Try a simpler scheme — sign change of real part of phase-rotated function.
# L_DH on critical line: phase rotation - the Hardy Z-function would need χ(s). Use real-valued sign changes
# of Im part after rotating by phase from functional equation.
# 
# Simpler approach: use sign change of REAL part of L_DH on a fine grid and refine to find any zero
# (real part vanishes), then check imag also small.

# Actually given time pressure, let's use simple sign changes of Re(L_DH(1/2+it)) on the fine grid.
re_vals = vals.real
sign_changes = []
for i in range(len(re_vals)-1):
 if re_vals[i] * re_vals[i+1] < 0:
 sign_changes.append((ts_scan[i], ts_scan[i+1]))

print(f"Re-part sign changes: {len(sign_changes)}")

# refine each via bisection on Re
dh_zeros = []
for t1, t2 in sign_changes[:200]:
 mp.mp.dps = 20
 a, b = mp.mpf(t1), mp.mpf(t2)
 fa = L_DH(mp.mpc(0.5, a)).real
 fb = L_DH(mp.mpc(0.5, b)).real
 if fa * fb >= 0:
 continue
 for _ in range(40):
 c = (a+b)/2
 fc = L_DH(mp.mpc(0.5, c)).real
 if fa*fc < 0:
 b, fb = c, fc
 else:
 a, fa = c, fc
 if abs(b-a) < mp.mpf('1e-10'):
 break
 t0 = (a+b)/2
 v = L_DH(mp.mpc(0.5, t0))
 # Check if Imag part also vanishes (true on-line zero)
 if abs(v) < 1e-3: # loose since we have rotational issue
 dh_zeros.append(float(t0))

print(f"Found {len(dh_zeros)} candidate on-line zeros via Re sign change")
print("first 10:", dh_zeros[:10])


TimeoutError: Code execution timed out after 110.0 seconds